# Vòng 60c: nén gallery ở quy mô ImageNet-1k

Gallery 1,28 triệu x 2048 (21 mảnh fp16, từ notebook `cure-in1k-extract-full`), phía truy vấn từ Dataset `cure-in1k-query`.
Các cấu hình giữ đủ 1,28 triệu vector (g0, bf16, ivf, hnsw) chạy **một tiến trình** vì gallery chiếm 5-10 GB;
các cấu hình thu nhỏ gallery chạy 4 tiến trình. Kết quả nén vào `/kaggle/working/edge_gallery_in1k_results.zip`.

In [ ]:
TASKS = ["grid"]          # "latency" cần CPU, chạy ở notebook riêng
BIG = ["g0", "bf16", "ivf4", "ivf16", "hnsw64"]   # giữ đủ 1,28 triệu vector: 1 tiến trình
SMALL_WORKERS = 4

In [ ]:
print("@@CELL 1@@", flush=True)
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
T0 = time.time()
def tick(m): print(f"[{(time.time() - T0) / 60:6.1f} phút] {m}", flush=True)

code_dir = next(t.parent for t in sorted(INPUT.rglob("pyproject.toml")) if 'name = "cure-ood"' in t.read_text(encoding="utf-8"))
REPO = WORK / "CURE"
if not REPO.exists():
    shutil.copytree(code_dir, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], capture_output=True, text=True)
print("faiss:", "ok" if r.returncode == 0 else "LỖI: " + r.stderr[-300:])

# Gom đặc trưng và mảnh gallery vào một thư mục phẳng (symlink, ko tốn chỗ).
FEAT = WORK / "features"; FEAT.mkdir(exist_ok=True)
for f in INPUT.rglob("*.np[yz]"):
    if f.suffix == ".npz" or f.name.startswith(("feats_", "labels_")):
        if not (FEAT / f.name).exists():
            (FEAT / f.name).symlink_to(f)
os.environ.update(CURE_FEATURES=str(FEAT), CURE_HF_AUTO="0")
os.chdir(REPO)
shards = sorted(FEAT.glob("feats_*.npy"))
print(open("VERSION.txt").read())
print(len(shards), "mảnh gallery,", sorted(p.name for p in FEAT.glob("*.npz")))

In [ ]:
print("@@CELL 2@@", flush=True)
import numpy as np
from cure_ood.data import Benchmark
from cure_ood.eval.runner import Experiment
EXP = "edge_gallery_in1k"
exp = Experiment.load(EXP)
b = Benchmark.load("imagenet1k")
sh = b.shards
print("gallery:", sh.size, "x", sh.dim, "| nhãn:", len(sh.labels), "lớp", len(np.unique(sh.labels)))
assert sh.size >= 1_281_167, f"thiếu ảnh train: {sh.size}"
print("C:", b.C.shape, "| luồng ID:", b.test_feats.shape, "| val OOD:", b.val_ood_data()[0].shape)
print("tập OOD:", {d: len(b.array(f"{d}_labels")) for d in b.domains})
tick("kiểm dữ liệu xong")

In [ ]:
print("@@CELL 3@@", flush=True)
ENV = {**os.environ, "OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}
LOGS = REPO / "results/logs"; LOGS.mkdir(parents=True, exist_ok=True)

def run_group(names, workers, tag):
    """Mỗi biến thể một tiến trình; tối đa `workers` chạy cùng lúc."""
    env = {**ENV, "OMP_NUM_THREADS": str(max(1, 4 // workers)), "MKL_NUM_THREADS": str(max(1, 4 // workers))}
    jobs = [[sys.executable, "-m", "cure_ood.cli", "run", EXP, "--benchmark", n] for n in names]
    pend, run, failed = list(enumerate(jobs)), {}, []
    while pend or run:
        while pend and len(run) < workers:
            i, cmd = pend.pop(0)
            log = open(LOGS / f"{tag}_{i}.log", "w")
            run[i] = (subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log, cmd[-1])
        for i, (p, log, name) in list(run.items()):
            if p.poll() is not None:
                log.close(); run.pop(i)
                tick(f"{tag}: xong {name} (mã {p.returncode})")
                if p.returncode:
                    failed.append(name)
        time.sleep(30)
    return failed

ALL = [x["name"] for x in exp.benchmarks]
big = [n for n in ALL if n.split("@")[1] in BIG]
small = [n for n in ALL if n not in big]
print("1 tiến trình:", big)
print(f"{SMALL_WORKERS} tiến trình:", small)

In [ ]:
print("@@CELL 4@@", flush=True)
if "grid" in TASKS:
    f1 = run_group(small, SMALL_WORKERS, "small")   # nhẹ, chạy trước để có kết quả sớm
    tick(f"nhóm nhỏ xong, lỗi: {f1}")
    f2 = run_group(big, 1, "big")
    tick(f"nhóm lớn xong, lỗi: {f2}")

In [ ]:
print("@@CELL 5@@", flush=True)
subprocess.run([sys.executable, "scripts/analyze_round60.py", "--results", f"results/{EXP}",
                "--pool", "imagenet1k", "--latency-base", "imagenet1k"], env=ENV)
out = WORK / f"{EXP}_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (REPO / "results").rglob("*"):
        if f.is_file() and (EXP in f.parts or "logs" in f.parts):
            z.write(f, f.relative_to(REPO))
print(out, round(out.stat().st_size / 1e6, 1), "MB")